# Milestone 4, Section 1.4: Simulated drift and continual updates

This notebook runs the continual-learning experiment of Section 1. A stream of labelled users arrives in seven drift intervals, from clean data to the severe shift that broke the Milestone 2 model in Milestone 3 (all 12 raw features +3 sd), and then back to clean data. Four update strategies are compared, each with its own deployed model:

| Strategy | What happens when the trigger fires |
|---|---|
| `static` | Nothing: the Milestone 2 model (v1) stays deployed |
| `retrain_new` | Full retrain on the new interval only (Milestone 3 approach) |
| `retrain_replay` | Full retrain on the new interval plus the replay buffer |
| `warm_replay` | **Selected:** warm-start boosting from the deployed model on the new interval plus the replay buffer; old trees frozen |

The trigger and the promotion gate are the Milestone 3 rules: an update starts only when the deployed model's Qini is below the floor in two consecutive windows of the interval's gate rows, and a candidate replaces the deployed model only when the lower end of the 95% paired bootstrap interval of its Qini gain is above zero. Every reported metric uses the untouched Milestone 2 test split, under the interval's drift (current) and without drift (retention).

Run all cells top to bottom (Runtime > Run all). Full data takes about 15-25 minutes on a Colab CPU runtime; set `FRAC = 0.2` for a quick check. Results are written to `outputs/continual/`.

In [ ]:
# SETUP: install the XGBoost version the saved model was written with, and make the pipeline importable.
# The package is looked for (1) next to this notebook, (2) in an uploaded Milestone_4_code.zip,
# (3) in a clone of the GitHub repository (pulled again if it already exists).
%pip install -q xgboost==3.4.1 psutil
import os, sys, glob, zipfile, subprocess
PKG = None
if os.path.isdir("uplift_pipeline"):
    PKG = "."
if PKG is None and glob.glob("Milestone_4_code*.zip"):
    zipfile.ZipFile(sorted(glob.glob("Milestone_4_code*.zip"))[-1]).extractall(".")
    PKG = "Milestone 4"
if PKG is None:
    if os.path.isdir("INSE-6450"):
        subprocess.run(["git", "-C", "INSE-6450", "pull", "-q"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/saopa28/INSE-6450.git"], check=True)
    PKG = os.path.join("INSE-6450", "Milestone 4")
if not os.path.isdir(os.path.join(PKG, "uplift_pipeline")):
    raise FileNotFoundError(f"uplift_pipeline not found in {PKG!r}. Push the 'Milestone 4' folder to GitHub, "
                            "or upload Milestone_4_code.zip to the Colab file panel and run this cell again.")
sys.path.insert(0, os.path.abspath(PKG))
import xgboost; print("xgboost", xgboost.__version__, "| pipeline from", os.path.abspath(PKG))

In [ ]:
import json
import numpy as np, pandas as pd
from IPython.display import display
from uplift_pipeline import load_data, split, load_v1, summary, CONTINUAL, OUT_DIR
from uplift_pipeline.continual import run_simulation, plot_results, baseline_efficiency, SCHEDULE, STRATEGIES

FRAC = 1.0            # share of rows to use; 1.0 reproduces the Milestone 2 split exactly
DATA = None           # None = download the engineered data from the repository release
OUT = OUT_DIR / "continual"

df = load_data(frac=FRAC) if DATA is None else load_data(DATA, frac=FRAC)
sp = split(df); del df
print({k: f"{len(v[2]):,} rows" for k, v in sp.items()})

In [ ]:
v1 = load_v1()                                  # Milestone 2 X-learner
Xte, tte, yte = sp["test"]
print("v1 test metrics:", {k: round(v, 6) for k, v in summary(yte, tte, v1.score(Xte)).items()})
print("v1 trees used for prediction:", v1.trees())
pd.DataFrame([{"hyperparameter": k, "value": v} for k, v in CONTINUAL.items()])

In [ ]:
res, upd = run_simulation(v1, sp, out_dir=OUT)

## Metrics before and after each update
`qini_before` is the deployed model when the interval arrives; `qini_after` is the model deployed after the interval's update decision. Both are on the test split under the interval's drift. `qini_original` is the deployed model on the test split without drift (retention of old knowledge).

In [ ]:
cols = ["step", "drift", "strategy", "psi_max", "monitor", "triggered", "updated", "promoted", "version_after",
        "qini_before", "qini_after", "auuc_before", "auuc_after", "uplift10_before", "uplift10_after", "qini_original", "auuc_original", "gate_ci_lo", "gate_ci_hi"]
pd.set_option("display.width", 250)
display(res.reindex(columns=cols).round(6))

In [ ]:
# Qini of each strategy's deployed model after every interval (current drift / original data)
cur = res.pivot(index=["step", "drift"], columns="strategy", values="qini_after")[list(STRATEGIES)]
ret = res.pivot(index=["step", "drift"], columns="strategy", values="qini_original")[list(STRATEGIES)]
print("Qini under the current drift"); display(cur.round(6))
print("Qini on the original (undrifted) test data"); display(ret.round(6))

## Efficiency of each promoted update
Update time, peak process memory (RSS) during the update, model size (stage-2 boosters used for scoring), number of trees, and model-only scoring latency for one request (batch 1) and a 256-row batch. Colab CPU runtime, so there is no VRAM figure.

In [ ]:
eff = pd.concat([pd.DataFrame([baseline_efficiency(v1, Xte)]), upd], ignore_index=True)
eff.to_csv(OUT / "efficiency.csv", index=False)
display(eff.round(3))

In [ ]:
fig = plot_results(res, upd, OUT)

In [ ]:
# Saved model versions (each has a manifest with its parent, training rows, gate result and file hashes)
for d in sorted((OUT / "models").iterdir()):
    m = json.loads((d / "manifest.json").read_text())
    print(f"{m['version']:24s} parent={m['parent']:22s} trees={m['trees']} size={m['scoring_size_mb']} MB")
print("\nFiles written:", sorted(p.name for p in OUT.iterdir()))